# The sKF bank with $q$ switched by an error detector

Follow-up of `kf-bank-q-sweep.ipynb` (2026-10-07). There, the sKF bank with AR($-0.9$) input and $\beta^* = 0.2$ had
the lowest floor at rest at $q = 10^{-2}$ ($-38.4$ dB) and the fastest recovery from the flip at $q = 0.1$ ($-15$ dB in
5 300 steps, against 15 931 at $10^{-2}$). Question: if each realisation's bank uses $q = 10^{-2}$ at rest and
$q = 0.1$ when its output error says something changed, does it get both?

$q$ only enters the mixing step, through $p_{ij}$. A $q_t$ that depends on $y_1, \dots, y_{t-1}$ only makes the
transition probabilities of step $t$ conditional on the past, so Algorithm 1 is unchanged apart from $p_{ij}(t)$.

**Choices (as simple as possible)**

| | |
|---|---|
| filter | the sKF bank of `kf-bank-q-sweep.ipynb`: six $\varepsilon$ from $10^{-9}$ to $10^{-4}$, Laplacian likelihood, $b_\eta = \mathrm{E}\lvert\eta_t\rvert$, $v_0 = 2$. Its `bank_step` with one line changed, so that each realisation has its own transition matrix |
| detector | per realisation, the running share ($\lambda = 0.01$, about 100 steps) of the steps where $\lvert e_t\rvert > b_\eta$, with $e_t = y_t - \boldsymbol{x}_t^{\mathsf T}\hat{\boldsymbol{w}}_{t-1}$ the bank's output error. On while the share is above 1/2. The $q$ of step $t$ uses the errors up to $t-1$ only |
| $q$ | $10^{-2}$ when the detector is off, $0.1$ when it is on |
| scenario | AR($-0.9$), $\beta^* = 0.2$, 5 dB: the change run of notebook 09 ($2N$ steps, the flip at $N$), 20 realisations, the same seeds |
| baselines | the same function with $q$ fixed at $10^{-2}$ and at $0.1$. They are also the control: they must give the sweep's numbers |
| measures | from the mean curve of the change run: the floor before the flip (mean over $[3N/4, N)$), the steps to $-15$ dB after the flip, the floor at the end (mean over the last quarter), the mean over the record. No rest run, so the floor before the flip is not the sweep's floor at rest (other noise draws) |

**Run time** about 3 min.

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'retina'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. Scenario and filters

Copied from `kf-bank-q-sweep.ipynb`, only the parts used here. One line changed in `bank_step`, marked.

In [ ]:
# === copied from kf-bank-q-sweep.ipynb section 2 (from bank-kf-excess-error.ipynb, commit f2ded06): the parts used
#     here, NOT edited ===
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run at rest; the change run has 2N
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
SNR_DB = 5.0                # nominal signal-to-noise ratio
VAR_THETA_0 = 2.0           # initial prior variance on each weight (ours; Leszek's is 1/M)
FS = 8000                   # sampling rate [Hz]

ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))


def noise_of(ar_a, beta=BETA):
    # Noise scale and b_eta = E|eta| at 5 dB for this input: notebook 09's lines, AR as a parameter.
    P_signal = ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
    var_eta = P_signal/10**(SNR_DB/10)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/beta) - gammaln(1/beta)))
    return scale_gg, scale_gg*np.exp(gammaln(2/beta) - gammaln(1/beta))


def signals(seed, ar_a, n, change_at=None, beta=BETA):
    # generate_signals / generate_signals_change of notebook 09 (commit 6e6dabc), with the AR
    # coefficient, the length and the instant of the flip as parameters. Same draws, same order.
    scale_gg, _ = noise_of(ar_a, beta)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(n + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    y = np.convolve(ho, x)[:n]
    if change_at is not None:
        y[change_at:] = -y[change_at:]
    eta = gennorm.rvs(beta, scale=scale_gg, size=n, random_state=rng)
    return x, y + eta


def stack(sigs):
    return np.array([s[0] for s in sigs]), np.array([s[1] for s in sigs])


def signal_power(ar_a):
    # h_o' R_xx h_o: the line of noise_of
    return ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
# === end of the copied block ===


# === copied from kf-bank-q-sweep.ipynb section 3 (from bank-skf-fkf.ipynb, commit 57bd568): the parts used here;
#     one line of bank_step changed, marked CHANGED; nothing else edited ===
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/07_skf_conjunto.ipynb, commit 9406077
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


LOG_2PI = np.log(2*np.pi)


def transition_matrix(K, q):
    """p_ij of eq. (eps.chain): q to each neighbour on the grid, 1 - q n_i to stay."""
    P = np.zeros((K, K))
    for i in range(K):
        neighbours = [j for j in (i - 1, i + 1) if 0 <= j < K]   # n_i = 1 at the ends, 2 inside
        P[i, neighbours] = q
        P[i, i] = 1 - q*len(neighbours)
    return P


def log_density_laplacian(e, sigma, b_eta):
    """log p_t(e) of eq. (bank.weights), from eq. (NL.density): log phi(u) + log(R(k-u) + R(k+u)) - log 2b."""
    u = e/sigma                                        # u = e / sigma
    k_t = sigma/b_eta                                  # k = sigma / b_eta
    return (-0.5*u**2 - 0.5*LOG_2PI                    # log phi(u)
            + np.logaddexp(log_mills(k_t - u), log_mills(k_t + u))   # log(R(k - u) + R(k + u))
            - np.log(2*b_eta))


def bank_step(W, V, log_mu, x_t, d_t, power, grid, log_P, chi_fn, log_density, scale):
    """One step of Algorithm 1 (V given) or Algorithm 2 (V = None) for B banks side by side.
    W (K, B, L) locations, V (K, B) variances, log_mu (K, B) log weights, x_t (B, L), d_t and power (B,).
    Returns the new W, V, log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    L = W.shape[-1]
    # mixing, eqs. (bank.mixing.prob) and (bank.mixing)
    log_joint = log_P + log_mu[:, None, :]               # CHANGED: log_P is (K, K, B), one matrix per bank
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    if V is not None:
        diff = W[:, None] - W_bar[None]                  # w_{t-1,i} - w_bar_{t,j}, (i, j, B, L)
        spread = np.einsum("ijbm,ijbm->ijb", diff, diff)/L
        V_bar = np.einsum("ijb,ijb->jb", mix, V[:, None] + spread)   # v_bar_{t,j}
        v_tilde = V_bar + grid[:, None]                  # v_tilde = v_bar + eps^(j)
    else:
        v_tilde = np.broadcast_to(grid[:, None], log_mu.shape)       # fKF: the fixed v^(j)
    # one sKF / fKF step per filter, eq. (bank.update)
    sigma = np.sqrt(v_tilde*power)                       # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + x_t*(chi/power)[..., None]
    if V is not None:
        V = v_tilde*(1 - slope/L)
    # weights, eq. (bank.weights), in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, V, log_mu, log_c, log_p, e/sigma


EPS_BANK = np.geomspace(1e-9, 1e-4, 6)     # the draft's remark, line 1614
# === end of the copied block ===


# copied from bank-skf-fkf.ipynb section 7 (commit 57bd568), NOT edited
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


print("scenario and filters ready")

## 3. The detector's level

How often $\lvert e_t\rvert > b_\eta$ at rest ($e_t = \eta_t$) and with a Gaussian excess error added, at a given
power below the signal's ($\boldsymbol{h}^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}$), by Monte Carlo. The
heavy-tailed noise is small most of the time, so the share rises well before the excess error reaches the noise
power. The detector, on above 1/2, therefore stays on until the excess error is about 15 dB below the signal.

In [ ]:
AR = -0.9
scale_gg, b_eta = noise_of(AR)
rng = np.random.default_rng(0)
eta = gennorm.rvs(BETA, scale=scale_gg, size=1_000_000, random_state=rng)
print(f"share of steps with |e_t| > b_eta, at rest: {np.mean(np.abs(eta) > b_eta):.2f}")
for rel_db in (-25, -20, -15, -10, -5):
    s = np.sqrt(signal_power(AR)*10**(rel_db/10))*rng.standard_normal(eta.size)
    print(f"   with an excess error {rel_db:>3} dB below the signal: {np.mean(np.abs(eta + s) > b_eta):.2f}")
del eta, s

## 4. The sKF bank with $q$ switched

`switched_bank` is `bank_batch` of `kf-bank-q-sweep.ipynb` for the sKF family without its options, plus the
detector. With `q_low = q_high` it is `bank_batch` at that $q$: same steps in the same order.

In [ ]:
LAMBDA = 0.01               # the detector's running share: about the last 100 steps
SHARE_ON = 0.5              # on while more than half of the recent |e_t| exceed b_eta


def switched_bank(X, D, b_eta, q_low, q_high, flip_at=N):
    """The sKF bank, B realisations side by side; at each step each realisation mixes with q_high if its detector is
    on, q_low otherwise. Returns the misalignment (B, n) and whether q_high was used (B, n)."""
    L, (B, n) = M, X.shape
    K = len(EPS_BANK)
    with np.errstate(divide="ignore"):                   # log 0 = -inf where p_ij = 0
        log_P_low, log_P_high = (np.log(transition_matrix(K, q))[:, :, None] for q in (q_low, q_high))
    W, V = np.zeros((K, B, L)), np.full((K, B), VAR_THETA_0)
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    share = np.zeros(B)                                  # running share of |e_t| > b_eta
    mis, high = np.empty((B, n)), np.zeros((B, n), dtype=bool)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        dw = w_hat - (ho if t < flip_at else -ho)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < L:
            continue
        on = share > SHARE_ON                            # from the errors up to t - 1
        high[:, t] = on
        e_out = D[:, t] - np.einsum("bm,bm->b", x_t, w_hat)          # the bank's output error
        power = np.einsum("bm,bm->b", x_t, x_t)
        W, V, log_mu, *_ = bank_step(W, V, log_mu, x_t, D[:, t], power, EPS_BANK,
                                     np.where(on, log_P_high, log_P_low), chi_laplacian, log_density_laplacian, b_eta)
        w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
        share += LAMBDA*((np.abs(e_out) > b_eta) - share)
    return mis, high


def summary(mis, high):
    curve = mis.mean(axis=0)
    after = high[:, N:]
    on_after = after.any(axis=1)
    first = np.argmax(after, axis=1)                     # first step on after the flip
    last = after.shape[1] - 1 - np.argmax(after[:, ::-1], axis=1)   # last step on after the flip
    return dict(curve=curve, share=high.mean(axis=0),
                before=10*np.log10(curve[3*N//4:N].mean()), to_15=time_to(curve, -15.0),
                end=10*np.log10(curve[N + 3*N//4:].mean()), mean=10*np.log10(curve.mean()),
                on_start=high[:, :N//2].sum(axis=1).mean(), on_rest=high[:, N//2:N].sum(axis=1).mean(),
                on_after=after.sum(axis=1).mean(), n_on=int(on_after.sum()),
                first=np.median(first[on_after]) if on_after.any() else -1,
                last=np.median(last[on_after]) if on_after.any() else -1)


print("switched bank ready")

## 5. Runs

The switched bank and the two fixed values of $q$, on the same 20 change runs. Control: the fixed runs against the
sweep's numbers from the mean curves (steps to $-15$ dB after the flip, mean over the record).

In [ ]:
start = time.time()
X, D = stack([signals(seed, AR, 2*N, change_at=N) for seed in range(R)])
RUNS = {"switched 1e-2 / 0.1": (1e-2, 0.1), "q = 1e-2": (1e-2, 1e-2), "q = 0.1": (0.1, 0.1)}
results = {}
for name, (q_low, q_high) in RUNS.items():
    results[name] = summary(*switched_bank(X, D, b_eta, q_low, q_high))
    print(f"{name} done [{time.time() - start:.0f} s]")
del X, D

# kf-bank-q-sweep.ipynb, sKF bank, AR(-0.9), beta* = 0.2, mean curves: steps to -15 dB after the flip, mean [dB]
SWEEP = {"q = 1e-2": (15931, -15.28), "q = 0.1": (5300, -16.52)}
print("\ncontrol (ours / sweep)")
for name, (steps, mean_db) in SWEEP.items():
    r = results[name]
    print(f"   {name:<9} -15 dB after the flip {r['to_15']} / {steps}, mean over the record {r['mean']:.2f} / {mean_db:.2f}")

print(f"\n{'run':<22}{'before flip':>13}{'-15 dB after':>14}{'end':>9}{'mean':>9}")
for name, r in results.items():
    print(f"{name:<22}{r['before']:>13.2f}{r['to_15']:>14d}{r['end']:>9.2f}{r['mean']:>9.2f}")
print("   floors and means in dB, from the mean curve; -15 dB after the flip in steps, -1 = never")

r = results["switched 1e-2 / 0.1"]
print(f"\nswitched run, detector on (mean over the 20 realisations): {r['on_start']:.0f} steps in [0, N/2), "
      f"{r['on_rest']:.0f} in [N/2, N), {r['on_after']:.0f} after the flip; after the flip it came on in "
      f"{r['n_on']} of 20 realisations, from step +{r['first']:.0f} to +{r['last']:.0f} (medians)")

## 6. Figure

Mean misalignment over the 20 realisations, the whole run and around the flip; below, the share of realisations
using $q = 0.1$ in the switched run. Every plotted number is in the table above.

In [ ]:
# Colours: categorical slots 1-3 of the dataviz skill's palette, checked with its validator (light surface: all pairs
# pass; the aqua is below 3:1 contrast on white, so the numbers are also in the table above).
COLOUR = dict(zip(RUNS, ["#2a78d6", "#eb6834", "#1baf7a"]))
LEVEL_GREY, GRID_GREY, INK = "#a3a29b", "#e1e0d9", "#0b0b0b"
t = (np.arange(2*N) - N)/FS                              # time from the flip [s]
fig, axes = plt.subplots(2, 2, figsize=(13, 6.5), sharex="col", gridspec_kw={"height_ratios": [3, 1]})
for col, xlim in enumerate([(t[0], t[-1]), (-0.25, 3.0)]):
    for name, r in results.items():
        axes[0, col].plot(t, 10*np.log10(r["curve"]), color=COLOUR[name], lw=1.3, label=name)
    axes[0, col].axhline(-15, color=LEVEL_GREY, lw=0.8)  # the level of the table
    axes[1, col].plot(t, results["switched 1e-2 / 0.1"]["share"], color=COLOUR["switched 1e-2 / 0.1"], lw=1.3)
    axes[1, col].set(xlim=xlim, ylim=(-0.05, 1.05), xlabel="time from the flip [s]")
axes[0, 0].set(ylabel="misalignment [dB]", title="whole run")
axes[0, 1].set(title="around the flip")
axes[1, 0].set(ylabel="share at q = 0.1")
axes[0, 0].legend(loc="upper right", frameon=False)
for ax in axes.flat:
    ax.axvline(0, color=INK, lw=0.8)                     # the flip
    ax.grid(color=GRID_GREY, lw=0.6)
    ax.set_axisbelow(True)
fig.suptitle("sKF bank, AR(-0.9), beta* = 0.2, 5 dB: q fixed or switched by the error detector")
fig.tight_layout()
plt.show()
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")

## 7. Findings

**Control: exact.** The fixed runs give the sweep's numbers: $-15$ dB after the flip in 15 931 and 5 300 steps, mean
over the record $-15.28$ and $-16.52$ dB.

**The switched bank gets the floor of $q = 10^{-2}$ but only part of the recovery of $q = 0.1$.**

| run | floor before the flip [dB] | $-15$ dB after the flip [steps] | floor at the end [dB] | mean over the record [dB] |
|---|---|---|---|---|
| switched $10^{-2}$ / $0.1$ | $-38.89$ | 11 730 | $-38.57$ | $-15.87$ |
| $q = 10^{-2}$ | $-38.64$ | 15 931 | $-38.11$ | $-15.28$ |
| $q = 0.1$ | $-27.19$ | 5 300 | $-27.51$ | $-16.52$ |

- The floor is 11 dB lower than at $q = 0.1$, and the recovery 26 % faster than at $q = 10^{-2}$, but 2.2 times slower
  than at $q = 0.1$. On the mean over the record (the draft's metric) it lies between the two: 0.59 dB better than
  $q = 10^{-2}$, 0.65 dB worse than $q = 0.1$. That mean is dominated by the large values after the flip, so the faster
  recovery of $q = 0.1$ counts for more than its 11 dB higher floor.
- **Why the recovery is not faster: the detector turns off too early.** It comes on 59 steps after the flip and turns
  off after 2 816 (medians over the 20 realisations), when the misalignment is still at about $-10$ dB (right panel).
  While it is on, the switched curve is the $q = 0.1$ curve; then it bends to the slope of $q = 10^{-2}$. With AR
  input the output error recovers long before the misalignment: the slow part is in the directions the coloured
  input barely excites, and those barely show in $e_t$. A detector on the output error cannot see that part.
- **No false alarms at rest:** 0 steps on in $[N/2, N)$ in every realisation. At the start it is on for about 1 800
  steps, during the first convergence.

**Not done.** A hold time (keep $q = 0.1$ for a fixed number of steps after the detector fires), or a lower detector
level: each adds or moves one parameter, and the result above says a detector on $e_t$ alone stops too early with AR
input. White input, Gaussian noise and the other banks are not run here.